# Milestone 3 — Công cụ gán nhãn thủ công
Hiển thị từng entry BibTeX và toàn bộ reference ứng viên cùng publication. Nhãn lưu ngay sau mỗi lựa chọn; bỏ qua vẫn là chưa gán nhãn.

Notebook đã trỏ vào bản dữ liệu thật tải ngày 24/11/2026. Đây là một dải 5.000 bài; phân công và quota hiện là tạm thời trước khi ghép đủ corpus nhóm. Tên member1–member3 cần thay bằng tên người thực tế khi bàn giao.

Cài dependency bằng `python -m pip install -r src/requirements.txt`. Trên Colab cần chuẩn bị source và dữ liệu tương ứng, sau đó sửa PROJECT_ROOT và DATA_ROOT.


In [ ]:
from pathlib import Path
import sys

# Trên Colab, đặt PROJECT_ROOT trỏ đến thư mục repo trên Drive.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src/matching').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src/matching').is_dir(), 'Đặt PROJECT_ROOT trỏ đến thư mục gốc của repo'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pyarrow as pa
from matching.candidate_pool import build_candidate_pool
from matching.labeling import run_labeling_session


## Chọn chế độ và người gán nhãn
`MODE='manual'`: dùng phân công 5 bài × 20 entry cho mỗi thành viên; không hiển thị kết quả heuristic.

`MODE='review'`: chỉ mở các entry heuristic đánh dấu cần kiểm tra; quyết định được lưu riêng và thuộc tập auto.

Điền REVIEWER bằng mã người thực tế. Mỗi người dùng file tiến độ riêng. Đầu vào phải giữ nguyên khi tiếp tục phiên. Có thể đặt DEMO=True để thử giao diện bằng dữ liệu giả lập.


In [ ]:
import json
from matching.labeling import pool_fingerprint
import pyarrow.parquet as pq

DEMO = False
MODE = 'manual'  # 'manual' hoặc 'review'
MEMBER = 'member1'  # member1 / member2 / member3
REVIEWER = ''  # Điền mã người gán nhãn; chưa chốt tên thì dùng mã thành viên tạm
DATA_ROOT = PROJECT_ROOT / 'runtime/matching/real-20261124'
BIBTEX_PATH = DATA_ROOT / 'input/m2/bibtex.parquet'
REFERENCES_PATH = DATA_ROOT / 'input/m1/references.parquet'
entry_keys = None

if DEMO:
    import tempfile
    demo_directory = tempfile.TemporaryDirectory(prefix='m3-labeling-demo-')
    progress = Path(demo_directory.name) / 'demo_labels.parquet'
    reviewer, publications = 'demo', None
    pool = build_candidate_pool(
        pa.Table.from_pylist([
            {'arxiv_id': 'demo-paper', 'citation_key': 'method',
             'title': 'A Method for Reference Matching', 'authors': ['Jane Doe'],
             'year': 2020, 'doi': '10.1234/demo'},
            {'arxiv_id': 'demo-paper', 'citation_key': 'missing',
             'title': 'An Unavailable Reference', 'authors': ['Alex Smith'], 'year': 2018},
            {'arxiv_id': 'demo-empty', 'citation_key': 'empty',
             'title': 'An Entry with No Candidates'},
        ]),
        pa.Table.from_pylist([
            {'arxiv_id': 'demo-paper', 'ref_arxiv_id': '2001.00001',
             'ref_title': 'A Method for Reference Matching', 'ref_authors': ['Jane Doe'],
             'ref_submission_date': '2020-01-02', 'ref_doi': '10.1234/demo'},
            {'arxiv_id': 'demo-paper', 'ref_arxiv_id': '2002.00002',
             'ref_title': 'Another Method', 'ref_authors': ['Other Author'],
             'ref_submission_date': '2020-02-03'},
        ]),
    )
else:
    if not REVIEWER.strip():
        raise ValueError('Điền REVIEWER trước khi gán nhãn dữ liệu thật')
    plan = json.loads((DATA_ROOT / 'dataset_plan.json').read_text(encoding='utf-8'))
    pool = build_candidate_pool(BIBTEX_PATH, REFERENCES_PATH)
    if plan['candidate_pool_sha256'] != pool_fingerprint(pool):
        raise ValueError('Dữ liệu thay đổi: cần tạo lại kế hoạch và kiểm tra nhãn cũ')
    reviewer = REVIEWER.strip()
    if MODE == 'manual':
        assigned = [a for a in plan['manual_assignments'] if a['owner'] == MEMBER]
        if not assigned:
            raise ValueError('MEMBER không có phân công trong kế hoạch')
        publications = [a['arxiv_id'] for a in assigned]
        entry_keys = [(a['arxiv_id'], k) for a in assigned for k in a['citation_keys']]
        progress = DATA_ROOT / 'manual' / f'{MEMBER}.parquet'
    elif MODE == 'review':
        queue = pq.read_table(DATA_ROOT / 'auto/review_queue.parquet').to_pylist()
        entry_keys = [(r['arxiv_id'], r['citation_key']) for r in queue]
        publications = sorted({r['arxiv_id'] for r in queue})
        progress = DATA_ROOT / 'review' / f'{reviewer}.parquet'
    else:
        raise ValueError('MODE phải là manual hoặc review')

print('Chế độ dữ liệu giả lập:', DEMO)
print('File tiến độ:', progress)
print('Số entry được chọn:', len(entry_keys) if entry_keys is not None else pool.entries.num_rows)
for row in pool.summary.to_pylist():
    if publications is None or row['arxiv_id'] in publications:
        print(row)


## Gán nhãn từng entry
- Nhập số thứ tự ứng viên để chọn đúng một reference và lưu ngay.
- Nhập n để xác nhận không có match, kể cả khi entry không có ứng viên.
- Nhập s để sang entry tiếp theo, giữ nguyên nhãn cũ nếu đã có.
- Nhập p để quay lại entry trước.
- Nhập c để xóa nhãn hiện tại.
- Nhập q để thoát.

Để trống hoặc nhập sai sẽ không lưu nhãn. Chạy lại cell bên dưới để tiếp tục từ entry chưa gán nhãn đầu tiên.

Năm phía reference được lấy từ ref_submission_date khi có dữ liệu. Trường thiếu được hiển thị là [thiếu dữ liệu].


In [ ]:
labels = run_labeling_session(
    pool, progress, reviewer=reviewer,
    publications=publications, entry_keys=entry_keys, input_fn=input,
)


## Kiểm tra tiến độ đã lưu
Chỉ những lựa chọn match hoặc xác nhận no-match mới được lưu. Entry chưa gán nhãn không có row trong file progress. Partition sẽ được gán ở bước chia tập sau.

Nhãn demo nằm trong thư mục tạm, tách khỏi tiến độ thật. Chạy lại cell thiết lập demo sẽ bắt đầu một demo mới.


In [ ]:
for row in labels.to_pylist():
    print(row)
print('Số entry đã gán nhãn:', labels.num_rows)
